# Bigtable

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/bigtable/bigtable.ipynb)

Google Cloud Bigtable is a wide-column NoSQL store for petabyte-scale, low-latency workloads such as time series, IoT and user profiles. It serves sparse rows through column families and splits data across tablets per node, so key and schema design decide performance. The questions below cover the data model, access patterns, replication and operations. The repetitive capability checklist from generic interview lists has been collapsed to one answer per topic, and the *Things to remember* sections near the end compress the limits and mistakes worth memorizing.

## What is Bigtable?

**Mental model.** Bigtable is a sparse, distributed, sorted map. A table maps one row key to one row, and a row holds column families, each with columns (qualifiers) and versioned cells. The map is sorted, so a range of row keys is one cheap scan.

**How it works.** Data is split into tablets (contiguous key ranges) served by a cluster of nodes. Nodes hold no durable data: SSTables live in Colossus, Google's distributed file system, so storage and compute scale and fail independently. This is the key difference from a single-node store.

**Gotcha.** Bigtable is not SQL and has no joins. If a query needs a join, an ad hoc `GROUP BY`, or a full-table aggregation, it belongs in BigQuery, not here.

Worked example: one table, one row key, four dimensions.

```text
row key:        acme#dev-00042#2026-10-04T12:00:00Z
column family:  r
column:         temperature
cell:           value=21.4, timestamp=2026-10-04T12:00:00Z
```

The same row key can carry `humidity`, `battery`, and later versions of `temperature`. Sparse means rows do not need the same columns, so the schema never has to be padded with nulls.

Common mistake: treating a table as relational and adding one row per column. Model one entity per row and put the varying fields in columns.

The cells below use `MiniBigtable`, a small pure-Python model of that sorted map (no Google Cloud account needed), to make row keys, versions, scans, GC and conditional mutations concrete.

In [ ]:
import bisect
from dataclasses import dataclass, field


@dataclass
class MiniBigtable:
    """A toy, single-process model of Bigtable's data model: a sorted map of
    row key -> family -> qualifier -> [(timestamp_micros, value), ...] newest first."""

    rows: dict = field(default_factory=dict)
    keys: list = field(default_factory=list)  # kept sorted, like tablets over a key space

    def set_cell(self, row_key: bytes, family: str, qualifier: bytes, value: bytes, ts: int) -> None:
        if row_key not in self.rows:
            bisect.insort(self.keys, row_key)
            self.rows[row_key] = {}
        versions = self.rows[row_key].setdefault(family, {}).setdefault(qualifier, [])
        versions.append((ts, value))
        versions.sort(reverse=True)  # reads see the newest version first

    def read_row(self, row_key: bytes, versions: int = 1) -> dict:
        row = self.rows.get(row_key, {})
        return {(f, q): cells[:versions] for f, cols in row.items() for q, cells in cols.items()}

    def read_rows(self, start: bytes, end: bytes, limit: int | None = None, versions: int = 1):
        """Half-open range scan [start, end) in key order: the only efficient read besides a point get."""
        lo, hi = bisect.bisect_left(self.keys, start), bisect.bisect_left(self.keys, end)
        for row_key in self.keys[lo:hi][:limit]:
            yield row_key, self.read_row(row_key, versions)

    def check_and_mutate(self, row_key, family, qualifier, expected, mutations, ts) -> bool:
        """CheckAndMutateRow: apply mutations atomically only if the newest cell equals expected."""
        current = self.read_row(row_key).get((family, qualifier))
        if (current[0][1] if current else None) != expected:
            return False
        for fam, qual, value in mutations:
            self.set_cell(row_key, fam, qual, value, ts)
        return True

    def increment(self, row_key, family, qualifier, delta, ts) -> int:
        """ReadModifyWriteRow: a server-side atomic increment of a 64-bit big-endian counter."""
        current = self.read_row(row_key).get((family, qualifier))
        value = int.from_bytes(current[0][1], "big", signed=True) if current else 0
        self.set_cell(row_key, family, qualifier, (value + delta).to_bytes(8, "big", signed=True), ts)
        return value + delta


bt = MiniBigtable()
T0 = 1_759_579_200_000_000  # 2026-10-04T12:00:00Z in microseconds
for minute, temp in enumerate([b"21.1", b"21.3", b"21.4"]):
    bt.set_cell(b"acme#dev-00042", "r", b"temperature", temp, T0 + minute * 60_000_000)
bt.set_cell(b"acme#dev-00042", "r", b"humidity", b"48", T0)  # sparse: other rows need not have it

print(bt.read_row(b"acme#dev-00042"))
assert bt.read_row(b"acme#dev-00042")[("r", b"temperature")][0][1] == b"21.4"
assert len(bt.read_row(b"acme#dev-00042", versions=3)[("r", b"temperature")]) == 3

## How do you design a good Bigtable row key?

A good Bigtable row key supports the query pattern and avoids hotspotting. Bigtable stores rows lexicographically by key, so monotonically increasing keys like raw timestamps can concentrate writes on one tablet. Use prefixes, salting, bucketing, or reversed timestamps when needed.

```text
customer_id#reverse_timestamp
region#bucket#device_id#timestamp
```

In [ ]:
import zlib
from datetime import datetime, timezone

MAX_TS = 9_999_999_999  # far-future sentinel for reversed timestamps
BUCKETS = 16


def row_key(tenant: str, device_id: str, ts: datetime) -> str:
    # A deterministic salt spreads hot tenants across tablets (Python's hash() is randomized per process);
    # a reversed, zero-padded timestamp sorts newest first.
    bucket = zlib.crc32(f"{tenant}#{device_id}".encode()) % BUCKETS
    reverse_ts = MAX_TS - int(ts.timestamp())
    return f"{bucket:02d}#{tenant}#{device_id}#{reverse_ts:010d}"


readings = [datetime(2026, 10, 4, 12, m, tzinfo=timezone.utc) for m in (0, 1, 2)]
keys = sorted(row_key("acme", "dev-00042", ts) for ts in readings)
print(keys)
assert keys[0] == row_key("acme", "dev-00042", readings[-1])  # newest reading sorts first

```text
00#acme#dev-00042#9999999999   -> newest first inside one device
00#acme#dev-00042#9999999900
01#globex#dev-00007#9999999998
```

**Mental model.** The row key is the only index, so it must encode the access pattern. Rows sort lexicographically by raw bytes. A prefix scan reads every row whose key starts with a prefix, in sorted order, from one contiguous key range.

**How it works.** Put the field you filter on first, then the field you sort by, then a unique suffix. Keep keys short (tens of bytes); long keys cost storage and cache on every row. Use fixed-width, zero-padded numbers so byte order matches numeric order: `2026-10-04` sorts correctly, `10/4/2026` does not.

**Gotcha.** A key that starts with a monotonically increasing value (a timestamp, a sequence, a `user-00001` counter) sends all new writes to one tablet end. That is the classic hotspot.

Worked example: queries drive the key.

```text
Query: latest readings for one device, newest first.
Key:   acme#dev-00042#9999999999   (reversed timestamp for newest-first)
Query: all devices in a region over a time window.
Key:   region#device_id#ts         (region first, so a region is one scan)
```

Common mistake: choosing the key from the entity shape instead of the query list. Write down the top three reads and writes first, then design the key to make each a prefix scan or a point lookup.

## When should you use Bigtable instead of BigQuery?

Use Bigtable for low-latency operational lookups, time-series access, and high-throughput key-value style workloads. Use BigQuery for analytical SQL, joins, scans, and aggregations. Bigtable is not a general data warehouse; it is optimized for serving rows by key/range.

+ Bigtable: fetch recent IoT readings for one device in milliseconds
+ BigQuery: calculate monthly usage by region across all devices

Many architectures use both: Bigtable for serving and BigQuery for analytics.

**Mental model.** Bigtable serves individual rows and narrow key ranges at low latency and high request rates. BigQuery scans columns across billions of rows for analytics. They answer different questions: "what is this user's state now" versus "what did all users do last quarter".

**How it works.** A Bigtable read is an index seek plus a few disk or cache reads, usually single-digit milliseconds. A BigQuery query reads columnar files in parallel and is billed by bytes scanned, so it prefers large scans and dislikes point lookups.

**Gotcha.** Running analytics on Bigtable forces full-table scans through a serving path and crowds out application traffic. Keep them apart, or read analytics through BigQuery federation or Data Boost.

Worked example: the same event stream, two serving needs.

```text
Serving:  get latest 20 events for user u-7781        -> Bigtable point + range read
Analytics: p95 latency by region for the last 90 days -> BigQuery scan
```

Common mistake: putting a hot dashboard on Bigtable "because the data is already there". Dashboards scan many rows and belong on a columnar copy.

Many architectures use both: Bigtable for serving, BigQuery for analytics, Change Streams or Dataflow to keep the copy fresh.

## How does Bigtable handle data storage?

**Mental model.** Bigtable stores a sparse, distributed, multidimensional sorted map. Each cell is `(row key, column family, column qualifier, timestamp) -> bytes`. Unset cells cost nothing, which is why wide and sparse rows are normal.

**How it works.** A write lands in a tablet node's commit log and in-memory memtable. When the memtable fills it flushes to an immutable SSTable in Colossus. A compaction later merges SSTables and drops versions the GC policy rejects. Nodes serve from cache, memtable, and SSTables; durable bytes always live in Colossus.

**Gotcha.** Storage is decoupled from nodes, so adding nodes does not add storage in the same way it would on a local-disk system. Size the cluster for throughput, and let storage grow independently.

Worked example: the write path in order.

```text
1. client sends mutation to the tablet node that owns the key range
2. node appends to the commit log (durability) and the memtable (readable now)
3. memtable flush -> SSTable in Colossus
4. compaction merges SSTables and applies the GC policy
```

Common mistake: assuming a delete frees space immediately. A delete writes a tombstone, and the bytes go away only when compaction removes the tombstoned versions.

## What is a tablet in Bigtable?

**Mental model.** A tablet is a contiguous range of row keys and the unit of distribution and load balancing, roughly the region of HBase. A table is a set of non-overlapping tablets that together cover the whole key space.

**How it works.** The service splits a tablet as it grows and merges neighbours when they shrink, so the number of tablets tracks data size rather than a fixed shard count. Tablets move between nodes as load changes, and clients are told which node serves which range.

**Gotcha.** The number of tablets is not something you set. You cannot pre-split or manually split a tablet in Cloud Bigtable, so a key that creates one huge range is fixed only by changing the key.

Worked example: key range to tablets.

```text
acme#dev-00001 .. acme#dev-09999  -> tablet A
acme#dev-10000 .. acme#dev-24999  -> tablet B
globex#...                        -> tablet C
```

Common mistake: assuming one tablet per table or one per node. The service decides, and it rebalances as the data grows.

## What consistency model does Bigtable provide?

Within a single cluster, Bigtable provides strong consistency: a read sees the result of every completed write. When an instance has clusters in more than one region, replication between them is asynchronous, so data converges across clusters instead of being immediately consistent. App profile routing decides which behavior a client gets: single-cluster routing for strict consistency, multi-cluster routing for lower-latency replicated reads.

**Mental model.** Within a single cluster, reads are strongly consistent: after a write commits, every later read sees it. Across clusters, replication is asynchronous, so a read from another cluster may briefly see an older version.

**How it works.** A normal app profile uses single-cluster routing and reads and writes the same cluster, so consistency is immediate. A multi-cluster app profile can route reads anywhere for lower latency, and accepts eventual consistency in exchange.

**Gotcha.** Do not mix the two expectations. If the application needs read-your-writes, pin it to one cluster, or read the row back from the cluster that accepted the write.

Worked example: choosing routing by need.

```text
read-your-writes required      -> single-cluster app profile
nearest-region reads required  -> multi-cluster app profile, eventual consistency
```

Common mistake: enabling multi-cluster routing to cut latency, then filing a bug that a just-written value is missing. That is the documented trade-off, not a fault.

## How does Bigtable handle schema changes?

**Mental model.** There is almost no schema to migrate. Adding a column family is an online admin operation, and adding a new column qualifier is just writing it. Existing rows simply do not have the new cell, which is what sparse means.

**How it works.** `cbt createfamily` or the client library adds a family without downtime. Removing a family drops its data. New qualifiers appear the first time a writer uses them, and readers must tolerate a missing cell.

**Gotcha.** Changing a GC policy is not retroactive in the way people expect; it takes effect as data is compacted, so old versions can linger briefly after you tighten the rule.

Worked example: adding a field with no migration.

```bash
cbt -project acme-prod -instance acme-bt createfamily device_readings battery
# writers start setting r:battery; older rows return no cell for it
```

Common mistake: writing migration jobs to backfill a new column. Read code should treat a missing cell as a default, and backfill only if analytics needs it.

## How does Bigtable handle data replication?

**Mental model.** Replication is per instance and per cluster. You add a second cluster in another zone or region, and Bigtable copies writes to it asynchronously. Each cluster holds a full copy of the table.

**How it works.** A write goes to one cluster first and is then replicated to the others. An app profile decides where a client reads and writes: one cluster for strong consistency, or any cluster for low-latency reads with eventual consistency. You can also route by cluster group to keep some data regional.

**Gotcha.** Replication is not a transaction across clusters. There is no cross-cluster atomic commit, and conflicting concurrent writes to the same cell resolve by last-write-wins on the server timestamp.

Worked example: two clusters, two app profiles.

```text
instance acme-bt
  cluster acme-bt-us-east1   (us-east1)
  cluster acme-bt-eu-west1   (europe-west1)
app profile "serving"  -> multi-cluster routing, reads nearest
app profile "billing"  -> single-cluster acme-bt-us-east1, read-your-writes
```

Common mistake: assuming a failover promotes a primary. There is no primary; clients reroute through the app profile.

## Does Bigtable support ACID transactions?

Bigtable gives ACID semantics at the row level: a row mutation, including a conditional read-modify-write, is atomic. Cross-row transactions are not the design center, so model around single-row atomicity instead of expecting relational-style multi-row transactions.

**Mental model.** Bigtable supports atomicity at the row level, not across rows. A single row mutation, which may touch many columns, applies as one unit. Conditional mutations give you compare-and-set on a row.

**How it works.** Use `CheckAndMutateRow` (Python: `row.commit()` after a conditional row, or the `conditional_row` API) to apply changes only if a predicate holds. This supports read-modify-write and optimistic concurrency within one row.

**Gotcha.** A batch that spans rows is not atomic. If one mutation in a batched write fails, others may already have applied, so retry logic must be idempotent.

## What is the recommended way to perform atomic row-level updates in Bigtable?

**Mental model.** Treat a row as the unit of consistency. Read the row, decide, and write it back with a conditional mutation so a concurrent writer cannot interleave. If the condition fails, re-read and retry.

**How it works.** `CheckAndMutateRow` takes a predicate filter and two mutation sets, one for true and one for false, and applies exactly one of them atomically. The read and the write are not a transaction, so you use the condition to detect interference and retry.

**Gotcha.** An unconditional read-modify-write is a lost-update bug under concurrency. Always guard it with a predicate, or use an aggregate family for counters.

Worked example: optimistic concurrency with a version cell, and an atomic counter.

In [ ]:
acct = b"acct-7781"
bt.set_cell(acct, "s", b"version", b"1", T0)
bt.set_cell(acct, "s", b"balance", b"100", T0)

# Optimistic concurrency: the write only lands if the version we read is still current.
ok = bt.check_and_mutate(acct, "s", b"version", expected=b"1",
                         mutations=[("s", b"balance", b"80"), ("s", b"version", b"2")], ts=T0 + 1)
stale = bt.check_and_mutate(acct, "s", b"version", expected=b"1",  # a second writer with a stale read
                            mutations=[("s", b"balance", b"50"), ("s", b"version", b"2")], ts=T0 + 2)
print("first write applied:", ok, "| stale write applied:", stale)
assert ok and not stale and bt.read_row(acct)[("s", b"balance")][0][1] == b"80"

# Counters: ReadModifyWriteRow increments atomically on the server, no lost updates.
for _ in range(5):
    total = bt.increment(b"acme#counter", "c", b"value", 1, T0)
assert total == 5

## How does Bigtable handle access control and security?

**Mental model.** Security has three layers: IAM roles for who may call which API, encryption for data at rest and in transit, and audit logging for who did what. There is no row-level SQL grant; access is per table through IAM.

**How it works.** Roles like `roles/bigtable.reader`, `roles/bigtable.user`, and `roles/bigtable.admin` are granted at the project, instance, or table level. Data is encrypted at rest by default, optionally with CMEK, and in transit with TLS. Data Access audit logs record reads and writes but are off by default.

**Gotcha.** IAM is not row-level. If two tenants must not see each other's rows, use separate tables or instances, or authorized views, not a column filter in the client.

Worked example: least privilege for a service account.

```bash
gcloud bigtable instances add-iam-policy-binding acme-bt \
  --member="serviceAccount:reader@acme.iam.gserviceaccount.com" \
  --role="roles/bigtable.reader"
```

Common mistake: granting project-wide `bigtable.admin` to an application so it can read one table. Grant the narrowest role at the narrowest resource.

## What is the maximum size of a row in Bigtable?

Bigtable enforces hard system limits on row key size, row size, and cell size (see the Cloud Bigtable quotas and limits page), and oversized rows hurt latency and write amplification. Design many small rows rather than a few huge ones, and keep large blobs in Cloud Storage with a reference in Bigtable.

**Mental model.** A row key is limited to 4 KiB, and the practical limit on a row is much smaller than a relational row. Bigtable is designed for many small rows, not a few huge ones.

**How it works.** A single cell value has a service maximum on the order of 100 MiB, but that ceiling is for rare blobs, not for routine data. Wide rows with thousands of columns can be written, but reading them is slow and they pin the tablet serving that key.

**Gotcha.** A row that grows without bound, such as one row per user holding all their events, becomes a hotspot and a slow read. Split time-ordered data across many rows.

Worked example: capping the row.

```text
bad:  row key = user-7781, qualifier = event#evt-0001  (unbounded row, one per user)
good: row key = user-7781#9999999999, qualifier = event  (one row per event)
```

Common mistake: storing a list in one cell and appending to it. Each append rewrites the whole value and creates a version; store one event per row instead.

## Does Bigtable support secondary indexes?

There is no classic relational secondary index, and the traditional answer was to duplicate data under a second row-key pattern. Current releases add asynchronous secondary indexes and continuous materialized views, plus GoogleSQL for Bigtable, so less of that duplication has to be hand-built.

**Mental model.** There is no built-in secondary index. The row key is the only ordered index. Any other lookup path is a second table that you keep in sync, or a BigQuery copy.

**How it works.** The standard pattern is an index table whose rows are `indexed_value#primary_key` and whose cells point back to the main row. Writes update both rows, ideally with Change Streams and Dataflow rather than a fragile dual write. Continuous materialized views and asynchronous secondary indexes are also available for some patterns.

**Gotcha.** An index table must handle the same hotspot rules as any table. Indexing a low-cardinality field, like `status`, puts millions of rows under a few prefixes.

Worked example: index by email.

```text
main table:  row = u-7781            family profile: email = "ada@acme.com"
index table: row = ada@acme.com#u-7781   family i: pk = "u-7781"
lookup:      read index row, then read main row
```

Common mistake: writing the index inline with the main write and ignoring partial failure. Use Change Streams or a conditional reservation so the two stay consistent.

## How does Bigtable handle backups and disaster recovery?

**Mental model.** A backup is a consistent, independent copy of a table that you create on demand and restore into a new table. Replication copies current state; a backup preserves a point in time that replication has already overwritten.

**How it works.** A backup is stored in a cluster and carries an optional expiration; by default it does not expire, and you delete it yourself. Restore creates a new table, then you point the application at it. You can also copy a backup to another region for cross-region recovery.

**Gotcha.** A backup is not continuous. Between two backups, a bad write or a dropped table loses everything since the last one, so schedule backups against your recovery point objective.

Worked example: create and restore.

```bash
gcloud bigtable backups create acme-daily-2026-10-04 \
  --instance=acme-bt --cluster=acme-bt-us-east1 --table=device_readings
gcloud bigtable tables restore device_readings_restored \
  --instance=acme-bt --cluster=acme-bt-us-east1 \
  --source=projects/acme-prod/instances/acme-bt/clusters/acme-bt-us-east1/backups/acme-daily-2026-10-04
```

Common mistake: restoring over the live table. Restore always targets a new table name; cut over deliberately.

## What are the considerations for choosing between Bigtable and other databases like Cassandra or MongoDB?

**Mental model.** Ask three questions: what is the access pattern, what consistency does the application require, and who operates the system. Cassandra and MongoDB are self-managed (or managed on another cloud); Bigtable is a managed Google service with a key-range model.

**How it works.** Bigtable and Cassandra both scale horizontally with a wide-column model, but Cassandra uses consistent hashing and tunable quorum consistency, while Bigtable uses sorted key ranges and single-cluster strong reads. MongoDB is a document store with secondary indexes and richer queries, which Bigtable lacks.

**Gotcha.** Choosing on features alone misses operations. If the team cannot run a Cassandra ring or a MongoDB replica set, a managed service changes the total cost more than any benchmark.

Worked example: fit by requirement.

```text
ordered range scans by key, managed      -> Bigtable
multi-cloud, tunable quorum, self-run    -> Cassandra
ad hoc queries and secondary indexes     -> MongoDB (or a relational store)
```

Common mistake: picking Bigtable for a document workload that needs many secondary indexes. Build those indexes explicitly or choose the document store.

## How does Bigtable handle compaction?

**Mental model.** Compaction merges immutable SSTables into fewer, larger ones and applies the garbage collection policy in the same pass. It is background work that reclaims space and keeps reads fast by limiting the number of files a read must check.

**How it works.** A flush produces a new SSTable from the memtable. Minor compactions merge small SSTables; major compactions rewrite a larger set and drop expired or tombstoned versions. Because GC happens during compaction, a tightened policy takes effect gradually.

**Gotcha.** Compaction consumes node CPU and disk bandwidth. A burst of writes produces many flushes, and the resulting compaction work competes with serving traffic, so size nodes for headroom, not for the average.

Worked example: from memtable to compacted SSTables.

```text
write -> memtable -> flush -> sstable-1
write -> memtable -> flush -> sstable-2
                 compaction merges sstable-1 + sstable-2, applies GC -> sstable-3
```

Common mistake: expecting a delete to reduce storage immediately. The tombstone and the old value remain until compaction merges the files that hold them.

## Can you explain how Bigtable manages garbage collection?

**Mental model.** GC is per column family and rule-based. A cell is kept or dropped by the rule when compaction processes it. There are two primitives, `max_num_versions` and `max_age`, and two combinators, `union` and `intersection`.

**How it works.** `max_num_versions` keeps the newest N versions of each cell. `max_age` keeps cells newer than a duration. `union` keeps a cell if any rule keeps it; `intersection` keeps it only if every rule keeps it. You set the rule with `cbt setgcpolicy` or the client library, and `cbt ls device_readings` shows the current policy.

**Gotcha.** GC runs at compaction, not at write time, so an expired cell can still be read until compaction reaches it. Read code must filter by timestamp if it needs strict freshness.

Worked example: the rules as JSON.

```text
keep newest 3 versions:      {"max_num_versions": 3}
keep 30 days:                {"max_age": "2592000s"}
newest 3 OR 30 days:         {"union": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
newest 3 AND 30 days:        {"intersection": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
```

The same policies from the command line:

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3 or maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3 and maxage=30d
```

Common mistake: using `union` when the goal is to cap storage. Union is the permissive rule; it keeps a cell if either rule keeps it. Use `intersection` to keep only cells that satisfy both.

In [ ]:
DAY_MICROS = 86_400 * 1_000_000


def gc_keep(versions, now, max_versions=None, max_age_days=None, mode="union"):
    """Apply a GC policy to one cell's versions (newest first), as compaction would."""
    keep = []
    for rank, (ts, value) in enumerate(versions):
        by_count = max_versions is not None and rank < max_versions
        by_age = max_age_days is not None and now - ts < max_age_days * DAY_MICROS
        rules = [r for r, active in ((by_count, max_versions), (by_age, max_age_days)) if active is not None]
        if (any(rules) if mode == "union" else all(rules)):
            keep.append((ts, value))
    return keep


now = T0
versions = [(now - d * DAY_MICROS, f"v{d}".encode()) for d in (0, 1, 2, 40, 90)]  # newest first
union = gc_keep(versions, now, max_versions=4, max_age_days=30, mode="union")
inter = gc_keep(versions, now, max_versions=4, max_age_days=30, mode="intersection")
print("newest 4 OR 30 days :", [v for _, v in union])   # permissive: keeps the 40-day-old v40
print("newest 4 AND 30 days:", [v for _, v in inter])   # caps storage
assert [v for _, v in union] == [b"v0", b"v1", b"v2", b"v40"]
assert [v for _, v in inter] == [b"v0", b"v1", b"v2"]

## How does Bigtable handle time-based data, such as event logs?

**Mental model.** Time is part of the key, not only a cell timestamp. Put the entity first and a reversed timestamp second so the newest events sort first and a prefix scan returns the latest N without reading history.

**How it works.** A key like `device-42#9999999999` gives newest-first ordering for one device. A GC policy of `max_age` then drops old events automatically, so storage stays bounded without a cleanup job. For queries across devices, put the grouping field first: `2026-10-04#device-42#ts`.

**Gotcha.** A raw ascending timestamp as the leading key field is the classic hotspot: every new event goes to the last tablet. Reverse it, salt it, or bucket by hour.

Worked example: latest-first scan.

```text
key = device-42#9999999990, device-42#9999999890, ...
scan prefix=device-42#, limit 10 -> newest ten rows, one range read
GC policy max_age=7d on the family -> older events expire at compaction
```

Common mistake: storing the timestamp only as the cell timestamp and keying by event id. Then "latest for this device" needs a full scan of the device's rows.

## Can you explain the role of a Bloom filter in Bigtable?

**Mental model.** A Bloom filter answers "this file definitely does not contain the key" cheaply, so reads skip SSTables that cannot hold the row. In HBase you choose filter types and tune them; in Cloud Bigtable the service manages this internally.

**How it works.** When a read looks for a row, Bigtable consults per-file metadata before touching the file. A negative answer skips a disk read, which matters when many SSTables cover a key range. The service decides the filter scope from the workload.

**Gotcha.** Bloom filters help point reads, not scans. A scan of many rows must read the data anyway, so a filter cannot make it cheaper.

Worked example: where the filter saves work.

```text
point read of device-42
  sstable-1: filter says no   -> skip
  sstable-2: filter says maybe -> read and find the row
```

Common mistake: hunting for a Bigtable Bloom filter setting. Unlike HBase, there is no `BLOOMFILTER` column-family attribute to set here.

## Does Bigtable support change data capture (CDC) for real-time data integration?

Bigtable supports change data capture with Change Streams: you enable a change stream on a table and read mutation records from it, usually with Dataflow. Managed templates stream changes to BigQuery or Pub/Sub, which is the modern path for keeping a warehouse, cache, or downstream service in sync with Bigtable.

**Mental model.** Change Streams is Bigtable's CDC feature. You enable it on a table, and Bigtable appends a record of every row change to a changelog that a stream processor reads within seconds.

**How it works.** A change stream can capture the latest value or all values. A Dataflow or Beam job reads the stream and writes it to BigQuery, Pub/Sub, or another sink. The retention window for streamed records is configurable and capped at a few days, so consumers must keep up.

**Gotcha.** A change stream is not a full history and not a transaction log. Records can be delivered more than once, so the sink must be idempotent.

Worked example: serving plus analytics.

```text
device_write -> Bigtable tables/device_readings (change stream enabled)
             -> Dataflow reads change stream -> BigQuery analytics
```

Common mistake: dual-writing to Bigtable and BigQuery from the application and hoping they stay consistent. Let the change stream be the single path.

## Can you explain how Bigtable handles data versioning?

**Mental model.** Every cell holds a list of timestamped versions, newest first. A write adds a version; it never overwrites history in place. A read returns the newest version unless you ask for more.

**How it works.** The server assigns the timestamp when you do not supply one, in microseconds. You can write your own timestamp for imports, replay, or out-of-order data. GC policy controls how many versions survive, and compaction enforces it.

**Gotcha.** A custom timestamp must respect the GC rules. If you import with old timestamps and the family uses `max_age`, the imported data can be garbage-collected almost immediately.

Worked example: version list on one cell.

```text
family r, qualifier temperature:
  ts 2026-10-04T12:00:00Z  "21.4"   <- read returns this
  ts 2026-10-04T11:59:00Z  "21.3"
  ts 2026-10-04T11:58:00Z  "21.1"
```

Common mistake: using a non-monotonic custom timestamp and then wondering why the newest value does not appear. Reads sort by timestamp descending.

## Can you explain how Bigtable handles range scans and filters?

**Mental model.** A scan walks rows in key order over a half-open range `[start, end)`. Filters then drop rows, columns, or cells before they reach the client, but the server still reads the key range you asked for.

**How it works.** Push the prefix into the key range so the scan starts and stops at the right place. Use cell filters to cut columns and versions, and a row limit to bound the result. In Python, `read_rows(start_key=..., end_key=...)` plus `row_filters` does this.

**Gotcha.** A filter is not a seek. Filtering by a value or a non-key column scans the whole range and throws most of it away. Design the key so the range itself is narrow.

Worked example: narrow range plus filters.

```bash
cbt -project acme-prod -instance acme-bt read device_readings prefix=acme#dev-00042# count=10
```

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=10,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
```

Common mistake: scanning a whole table and filtering by `device_id` in the client. Put `device_id` in the key prefix so the server does the narrowing.

In [ ]:
# Load 3 devices x 5 readings with reversed-timestamp keys, then scan one device's prefix.
scan_bt = MiniBigtable()
for device in ("dev-00041", "dev-00042", "dev-00043"):
    for minute in range(5):
        key = f"acme#{device}#{MAX_TS - (1_759_579_200 + minute * 60):010d}".encode()
        scan_bt.set_cell(key, "r", b"temperature", f"{20 + minute}.0".encode(), T0 + minute)

# "#" is 0x23 and "$" is 0x24, so [prefix#, prefix$) covers exactly the prefix.
latest = list(scan_bt.read_rows(b"acme#dev-00042#", b"acme#dev-00042$", limit=2))
for key, cells in latest:
    print(key.decode(), cells[("r", b"temperature")][0][1].decode())
assert len(latest) == 2
assert latest[0][1][("r", b"temperature")][0][1] == b"24.0"  # newest first, no sort needed

## Tuning row keys for even load and locality

**Mental model.** Two goals pull in opposite directions. Even load wants writes spread across the whole key space. Locality wants the rows one query needs to sit next to each other. A good key satisfies both for the queries that matter, and accepts a trade-off for the rest.

**How it works.** Diagnose first. Key Visualizer shows per-prefix traffic; a bright horizontal band on a prefix means a hotspot. Then choose a fix: field promotion moves a high-cardinality field to the front, salting prepends a fixed-width bucket, hashing spreads a single hot key, and a reversed timestamp turns "latest first" into a prefix scan.

**Gotcha.** Salting destroys free range scans over the salted field, because the salt reorders rows. Add it only to the leading, hottest part of the key, and keep the rest sorted.

Worked example: read the symptom, then fix the key.

```text
symptom: writes all land on the last tablet, node CPU is pegged
cause:   key = 2026-10-04T12:00:00Z#device-42 (ascending timestamp first)
fix:     key = 07#device-42#9999999999 (16 salt buckets, reversed timestamp)

symptom: one user row takes most reads, one tablet is hot
cause:   key = CONFIG#global-flags
fix:     key = 01#CONFIG#global-flags plus an application cache
```

A hotspot check is simple arithmetic: compute the share of writes per key prefix and compare the largest share to an even split.

```python
from collections import Counter

counts = Counter(k.split("#")[0] for k in ["00#d1", "00#d2", "00#d3", "03#d4", "03#d5"])
total = sum(counts.values())
worst = max(counts.values()) / total
print(f"worst prefix share = {worst:.0%} across {len(counts)} prefixes")
```

Common mistake: salting and then querying the old unsalted prefix. Reads must use the same fan-out as writes.

## Tuning GC policies and version counts

**Mental model.** GC policy is a storage-versus-read-cost dial. More retained versions mean more bytes, more compaction work, and slower wide reads. Fewer versions mean cheaper storage and faster reads, but less history. Set the rule per column family from the retention the product actually needs.

**How it works.** `max_num_versions` caps versions per cell. `max_age` caps age. `union` keeps a cell if either rule keeps it, so it is the permissive combination. `intersection` keeps a cell only if both keep it, so it is the restrictive one. The rule is enforced at compaction.

**Gotcha.** The default policy is not a storage cap. A table that appends to the same cell every second can retain far more than expected until you set a version limit.

Worked example: choose per family, not per table.

```text
family profile: {"max_num_versions": 1}          # only current state, cheapest
family events:  {"max_age": "2592000s"}          # 30 days of history
family audits:  {"max_num_versions": 12}         # last 12 changes, no age cap
combined:       {"union": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
```

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings events maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings profile maxversions=1
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: using `union` to try to cap storage. Union keeps cells that any rule keeps, so it retains more; use `intersection` to bound bytes.

## Tuning cluster sizing, nodes, and app profiles

**Mental model.** Nodes buy throughput; storage grows on its own. Size the cluster from peak requests per second, add headroom for compaction, and use app profiles to route that traffic to the right cluster with the right consistency.

**How it works.** Plan on the documented per-node ceiling of roughly 10,000 simple reads or writes per second, and target well below it (about 60 to 70 percent) so compaction and spikes have room. Autoscaling adjusts between a minimum and maximum against a target utilization. Choose SSD for latency-sensitive workloads. An app profile selects single-cluster routing (read-your-writes) or multi-cluster routing (nearest replica, eventual consistency), and can be limited to a cluster group.

**Gotcha.** Read and write load share the same nodes, so take the larger of the two estimates, not the sum. And size every cluster for full peak if you expect seamless failover.

Worked example: node count from peak throughput.

```text
peak reads  = 50,000/s   peak writes = 20,000/s
usable per node = 10,000 * 0.7 = 7,000/s
read nodes  = ceil(50,000 / 7,000) = 8
write nodes = ceil(20,000 / 7,000) = 3
cluster     = 8 nodes (the larger requirement)
```

```bash
cbt -project acme-prod -instance acme-bt createcluster acme-bt-eu \
  --zone=europe-west1-b --num-nodes=8
cbt -project acme-prod -instance acme-bt createappprofile acme-bt eu-serving \
  --route-any
cbt -project acme-prod -instance acme-bt createappprofile acme-bt ledger \
  --route-to=acme-bt-us-east1
```

Common mistake: sizing on average QPS. Peak-to-average ratios above three are normal in serving workloads.

```python
from google.cloud import bigtable
from google.cloud.bigtable import enums

instance = bigtable.Client(project="acme-prod", admin=True).instance("acme-bt")
# Multi-cluster routing: nearest healthy cluster, eventually consistent across clusters.
instance.app_profile("eu-serving", routing_policy=enums.RoutingPolicyType.ANY).create()
# Single-cluster routing: read-your-writes on one cluster.
instance.app_profile("ledger", routing_policy=enums.RoutingPolicyType.SINGLE).create()
```

## Tuning reads and writes: batching, filters, and caching

**Mental model.** Reads are cheapest when they are point lookups or narrow key ranges that ask for few versions and few columns. Writes are cheapest when the key spreads them and GC bounds the versions they create. Everything else is overhead you can design away.

**How it works.** Bound every scan with a start key, an end key, and a limit. Use filters to cut columns and versions: a cells-per-column limit, a column-qualifier regex, a timestamp range. Batch writes with `mutate_rows`, but keep batches small enough to avoid timeouts and make retries idempotent. For very large read jobs, use Data Boost or a replica so serving traffic is untouched.

**Gotcha.** A row filter is applied after the server reads the key range. Filtering on a value does not narrow the scan; only the key range does.

Worked example: bound reads, then federate for analytics.

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()
```

For analytical questions, query Bigtable from BigQuery through an external table and keep a row-key filter so predicates push down.

```sql
SELECT rowkey, MAX(temperature) AS max_temp
FROM analytics.device_readings
WHERE rowkey LIKE 'acme#dev-00042%'
GROUP BY rowkey;
```

Common mistake: reading a wide row and filtering columns in the application. A qualifier filter drops those bytes before they cross the network.

## Things to remember: key numbers and limits

| Topic | Number / fact | Why it matters |
|---|---|---|
| Row key | max 4 KiB, byte-sorted across tablets | the row key IS the index; no secondary indexes |
| Row/cell size | design many small rows; a cell has a service maximum and oversized rows hurt latency and write amplification | unbounded rows per entity (e.g. one row per user, all events) is the classic scaling failure |
| Transactions | ACID at the single-row level only, including conditional read-modify-write | model aggregates and counters per row; cross-row atomicity does not exist |
| Consistency | single-cluster reads are strongly consistent; multi-cluster replication is eventually consistent | pick app profile routing per read consistency need |
| Versioning | each cell keeps multiple timestamped versions; GC policies bound age and count | versions are not time travel for analytics; GC them deliberately |
| Full-text / aggregations / auto-indexing | not provided | pair with Elasticsearch for search, BigQuery for analytics |
| Throughput | scales per node with autoscaling; hot rows do not split | a hot row key caps throughput no matter how many nodes |

## Things to remember: semantics to memorize

* **The row key decides everything.** Tablets split on key ranges, so a monotonically increasing key (timestamp-first) writes to one tablet at a time — hotspotting by design. Hash or reverse timestamps deliberately.
* **Reads are point-lookup or prefix scan.** Anything beyond that is a full-table scan: there is no query planner, no secondary index, no ad-hoc analytics.
* **GC policies are your TTL.** Age and version-count policies keep time-series rows bounded; without them every cell grows forever.
* **App profiles route traffic.** Single-cluster routing for strong consistency, multi-cluster for failover — the choice is per workload, not per table.
* **Compaction rewrites storage.** Write amplification during compactions is normal; watch latency during windows, and let GC policies shrink what needs re-writing.

## Things to remember: mistakes that keep coming back

* Timestamp-first row keys — all writes hit the newest tablet; salt, hash, or reverse the timestamp (`Long.MAX_VALUE - ts`).
* One unbounded row per entity — flip the time component into the key so a row stays one event.
* Expecting cross-row transactions or foreign keys — remodel around single-row atomicity, or use a relational store.
* Using Bigtable for ad-hoc analytics or search — it is neither; it is the high-throughput lookup layer beside BigQuery and Elasticsearch.
* Letting versions accumulate with no GC policy — storage and scan cost grow with every write.
* Sizing the cluster while hotspotting — more nodes cannot rescue a bad row key; fix the key first.
* Treating replicated clusters as strongly consistent — use a single-cluster app profile where reads must be current.

## Real world use cases for Bigtable

### 1. IoT telemetry serving

Millions of devices write readings every few seconds; the application shows the latest readings per device and a downsampled history.

The same operations with the real `google-cloud-bigtable` client (needs an instance, so shown for reading):

```python
from google.cloud import bigtable
from google.cloud.bigtable import column_family, row_filters

client = bigtable.Client(project="acme-prod", admin=True)
table = client.instance("acme-bt").table("device_readings")
table.create()
table.column_family("r", gc_rule=column_family.MaxAgeGCAgeRule(7 * 24 * 60 * 60)).create()  # 7-day TTL

row = table.direct_row(b"acme#dev-00042#9999999999")
row.set_cell("r", b"temperature", b"21.4")
row.commit()

rows = table.read_rows(start_key=b"acme#dev-00042#", end_key=b"acme#dev-00042$", limit=50,
                       filter_=row_filters.CellsColumnLimitFilter(1))
```

Why Bigtable: millisecond key lookups at very high write rates, with TTL enforced by garbage collection instead of a cleanup job. Salt or bucket the key prefix when one device or tenant dominates traffic.

### 2. User activity and message history

A messaging product stores per-conversation history and a per-user inbox. Reads are prefix scans ordered by time.

In [ ]:
# "The last 50 messages" is a bounded prefix scan when the key is conv#reversed_ts.
chat = MiniBigtable()
for n in range(120):
    chat.set_cell(f"conv-7781#{MAX_TS - (1_759_579_200 + n):010d}".encode(), "m", b"body", f"msg {n}".encode(), T0 + n)
page = list(chat.read_rows(b"conv-7781#", b"conv-7781$", limit=50))
assert len(page) == 50 and page[0][1][("m", b"body")][0][1] == b"msg 119"
print("newest:", page[0][1][("m", b"body")][0][1], "| oldest on page:", page[-1][1][("m", b"body")][0][1])

Why Bigtable: a sorted row key gives "the last 50 messages" with a range scan and no secondary index, while appends stay cheap. Model the "all conversations for a user" query as a second row-key layout or an asynchronous secondary index.

### 3. Online feature store for ML serving

A fraud model needs 30-day aggregates per card at prediction time.

```text
row key: card#c-9182#feature#velocity_24h
columns: f:value, f:updated_at
write:   conditional mutation so an older write cannot overwrite a newer value
read:    one row per feature at scoring time (sub-10 ms p99)
train:   export or stream the same features to BigQuery for offline training
```

Why Bigtable: the serving path needs predictable latency at high QPS, which is exactly a row-key lookup. Do not scan Bigtable to build training sets - use BigQuery or Parquet, and keep both paths fed by one feature definition to avoid training/serving skew.

### 4. Real-time bidding and profile lookup

Ad tech keeps a user profile that every bid request reads, and reads must be fast in every region.

```text
row key:  bucket#user_id          (bucket spreads hot users across tablets)
families: profile (small, rarely changing), events (append-only, GC by age)
routing:  multi-cluster app profile so bids read the nearest replica
```

Why Bigtable: high read throughput with replicated low-latency reads. Multi-cluster routing accepts eventual consistency for profile updates, which bidding tolerates, in exchange for serving reads close to the user.

### 5. Operational serving plus BigQuery analytics

The same data serves the application and the analyst. Change Streams feed the analytical copy instead of a dual write.

```text
Bigtable (serving) --Change Streams--> Dataflow --> BigQuery (analytics) --> BI
                          |
                   Data Boost for large read jobs without competing with serving traffic
```

```sql
-- analytical questions run in BigQuery, not against the serving table
SELECT DATE(event_ts) AS ds,
       COUNT(*) AS events,
       APPROX_QUANTILES(latency_ms, 100)[OFFSET(95)] AS p95
FROM analytics.events_enriched
GROUP BY ds
ORDER BY ds;
```

Why Bigtable: the serving table stays optimized for key lookups, analytics runs on a columnar copy, Change Streams remove the fragile dual write, and Data Boost isolates heavy scans from production traffic.

## Bigtable exercises

These use only the standard library to model key, sizing, and GC decisions. Treat the asserted values as checks on the formula, not benchmarks.

### Try it 1. Design a row key from access patterns

**Task.** Build a row key for the queries "latest reading for one device, newest first" and "all readings for a device in a time window". The key must be `salt#device_id#reverse_ts` with a two-digit salt, and the reversed timestamp must sort newest first.

**Hint.** Use a far-future sentinel and subtract the epoch milliseconds: `9_999_999_999_999 - ts_ms`. Format with `%013d`, and derive the salt with `hashlib.md5(device_id.encode())` modulo the bucket count.

**Expected result.** Every key starts with a two-digit salt below 16, keys for one device sort newest first, and a prefix scan returns them in that order.

In [ ]:
import hashlib

BUCKETS = 16
MAX_TS = 9_999_999_999_999


def row_key(device_id, ts_ms):
    salt = int(hashlib.md5(device_id.encode()).hexdigest(), 16) % BUCKETS
    return f"{salt:02d}#{device_id}#{MAX_TS - ts_ms:013d}"


base = 1_700_000_000_000
timestamps = [base + offset for offset in (120_000, 60_000, 0)]  # newest first
keys = [row_key("dev-00042", ts) for ts in timestamps]
for key in keys:
    print(key)

salts = [key.split("#")[0] for key in keys]
print("distinct salts:", sorted(set(salts)))
print("newest first:", keys == sorted(keys))

assert all(len(salt) == 2 and 0 <= int(salt) < BUCKETS for salt in salts)
assert keys == sorted(keys)
assert keys[0].endswith(f"{MAX_TS - (base + 120_000):013d}")
assert len(set(keys)) == 3
print("ok")

### Try it 2. Compute storage growth from rows, versions, and cell size

**Task.** A table has 500 million rows, an average key plus value size of 120 bytes, keeps 3 versions per cell, and replicates to 2 clusters. Compute raw stored bytes, the GiB figure, and the bytes after a 3x compression ratio.

**Hint.** Raw bytes are `rows * cell_bytes * versions * clusters`. Compression divides by the ratio. Use `1024 ** 3` for GiB and round only when printing.

**Expected result.** `360,000,000,000` raw bytes, about `335.3 GiB`, and `120,000,000,000` compressed bytes.

In [ ]:
ROWS = 500_000_000
CELL_BYTES = 120
VERSIONS = 3
CLUSTERS = 2
COMPRESSION_RATIO = 3.0

raw_bytes = int(ROWS * CELL_BYTES * VERSIONS * CLUSTERS)
compressed_bytes = int(raw_bytes / COMPRESSION_RATIO)

print(f"raw bytes        = {raw_bytes:,}")
print(f"raw GiB          = {raw_bytes / 1024 ** 3:.1f}")
print(f"compressed bytes = {compressed_bytes:,}")

assert raw_bytes == 360_000_000_000
assert round(raw_bytes / 1024 ** 3, 1) == 335.3
assert compressed_bytes == 120_000_000_000
print("ok")

### Try it 3. Estimate node count from throughput targets

**Task.** Peak load is 50,000 reads per second and 20,000 writes per second. A node handles about 10,000 simple reads or writes per second, and you plan to use 70 percent of that. Compute the nodes needed for reads, for writes, and the cluster size.

**Hint.** Use `math.ceil` on `peak / (per_node * target_utilisation)`. Reads and writes share nodes, so the cluster needs the larger of the two numbers, not the sum.

**Expected result.** 8 read nodes, 3 write nodes, and a cluster of 8 nodes.

In [ ]:
import math

PEAK_READS = 50_000
PEAK_WRITES = 20_000
PER_NODE = 10_000
TARGET = 0.7

usable_per_node = PER_NODE * TARGET
read_nodes = math.ceil(PEAK_READS / usable_per_node)
write_nodes = math.ceil(PEAK_WRITES / usable_per_node)
cluster_nodes = max(read_nodes, write_nodes)

print(f"usable per node = {usable_per_node:,.0f} operations/s")
print(f"read nodes      = {read_nodes}")
print(f"write nodes     = {write_nodes}")
print(f"cluster nodes   = {cluster_nodes}")

assert usable_per_node == 7_000.0
assert read_nodes == 8
assert write_nodes == 3
assert cluster_nodes == 8
print("ok")

### Try it 4. Size a scan from row count and filter selectivity

**Task.** A range scan covers 8 million rows of 200 bytes each. A qualifier filter keeps 1 column in 40, and a version filter keeps 1 version in 5. Compute rows returned, bytes read before filters, and bytes returned after filters.

**Hint.** Rows returned is `rows * column_selectivity * version_selectivity`. Bytes before filters use the full row count; bytes after use the returned rows. Use integer division only when the arithmetic is exact.

**Expected result.** 40,000 rows returned, `1,600,000,000` bytes read before filters, and `8,000,000` bytes returned.

In [ ]:
ROWS = 8_000_000
ROW_BYTES = 200
COLUMN_SELECTIVITY = 1 / 40
VERSION_SELECTIVITY = 1 / 5

rows_returned = int(ROWS * COLUMN_SELECTIVITY * VERSION_SELECTIVITY)
bytes_before = ROWS * ROW_BYTES
bytes_after = rows_returned * ROW_BYTES
reduction = bytes_before // bytes_after

print(f"rows scanned    = {ROWS:,}")
print(f"rows returned   = {rows_returned:,}")
print(f"bytes before    = {bytes_before:,}")
print(f"bytes returned  = {bytes_after:,}")
print(f"reduction       = {reduction}x")

assert rows_returned == 40_000
assert bytes_before == 1_600_000_000
assert bytes_after == 8_000_000
assert reduction == 200
print("ok")

### Try it 5. Detect a hotspot from key-prefix distribution

**Task.** Given request counts per key prefix, compute the total, the even share, the busiest prefix, its ratio to the even share, and the prefixes above twice the even share.

**Hint.** The even share is `total / len(counts)`. The ratio is `count / even_share`. List comprehension over the counts finds the offenders.

**Expected result.** Total `134,500`, even share `16,812.5`, busiest `00` at about `2.97x`, and `["00"]` above the 2.0x threshold.

In [ ]:
counts = {
    "00": 50_000,
    "01": 20_000,
    "02": 18_000,
    "03": 12_000,
    "04": 10_000,
    "05": 9_000,
    "06": 8_000,
    "07": 7_500,
}

total = sum(counts.values())
even_share = total / len(counts)
ratios = {prefix: count / even_share for prefix, count in counts.items()}
busiest = max(ratios, key=ratios.get)
hot = [prefix for prefix, ratio in ratios.items() if ratio > 2.0]

print(f"total requests = {total:,}")
print(f"even share     = {even_share:,.1f}")
print(f"busiest        = {busiest} at {ratios[busiest]:.2f}x")
print("prefixes above 2.0x:", hot)

assert total == 134_500
assert even_share == 16_812.5
assert round(ratios[busiest], 2) == 2.97
assert hot == ["00"]
print("ok")

### Try it 6. Plan a GC policy from retention days and versions

**Task.** A family must keep 30 days of history but never more than 5 versions per cell. Write a function that classifies a proposed policy as `union`, `intersection`, or `single`, and return the tighter combination for the requirement.

**Hint.** `union` keeps a cell if any rule keeps it. `intersection` keeps it only if all rules keep it, which is what "no more than 5 versions AND no older than 30 days" means. Convert days to seconds with `days * 86_400`.

**Expected result.** The combined policy is `intersection`, `max_age_seconds` is `2,592,000`, and the version cap is `5`.

In [ ]:
def plan_gc(retention_days, max_versions):
    if retention_days and max_versions:
        policy = "intersection"
    elif retention_days or max_versions:
        policy = "single"
    else:
        policy = "none"
    return {
        "policy": policy,
        "max_age_seconds": retention_days * 86_400 if retention_days else None,
        "max_num_versions": max_versions,
    }


plan = plan_gc(retention_days=30, max_versions=5)
print(plan)

assert plan["policy"] == "intersection"
assert plan["max_age_seconds"] == 2_592_000
assert plan["max_num_versions"] == 5
assert plan_gc(0, 3)["policy"] == "single"
assert plan_gc(7, 0)["max_age_seconds"] == 604_800
assert plan_gc(0, 0)["policy"] == "none"
print("ok")

### Try it 7. Model salting fan-out

**Task.** A single hot key takes 64,000 writes per second. Model fan-out with 4, 8, and 16 salt buckets, and pick the smallest bucket count that keeps each salted key under 10,000 writes per second.

**Hint.** Writes per bucket are `total / buckets`. Iterate the candidate bucket counts in ascending order and stop at the first that satisfies the limit.

**Expected result.** 16,000 then 8,000 then 4,000 writes per bucket, and a chosen fan-out of 8 buckets.

In [ ]:
TOTAL_WRITES = 64_000
PER_KEY_LIMIT = 10_000

fan_out = {}
for buckets in (4, 8, 16):
    per_key = TOTAL_WRITES / buckets
    fan_out[buckets] = per_key
    print(f"{buckets:>2} buckets -> {per_key:,.0f} writes/s per salted key")

chosen = next(b for b in (4, 8, 16) if TOTAL_WRITES / b <= PER_KEY_LIMIT)
print("chosen fan-out:", chosen)

assert fan_out[4] == 16_000.0
assert fan_out[8] == 8_000.0
assert fan_out[16] == 4_000.0
assert chosen == 8
print("ok")

### Try it 8. Plan a backup schedule from a recovery point objective

**Task.** A table receives 2,000 writes per second. The recovery point objective allows losing at most 15 minutes of data, and a backup takes 4 minutes. Decide whether a 10-minute backup interval meets the objective, and compute the worst-case data loss.

**Hint.** Worst-case loss is `(interval + backup_duration)` minutes of writes, because a crash during a backup can lose the interval plus the in-progress backup window. Compare that to the 15-minute objective in seconds.

**Expected result.** Worst-case loss is 14 minutes (`1,680,000` writes), a 10-minute interval meets the objective, and a 15-minute interval does not.

In [ ]:
WRITES_PER_SECOND = 2_000
RPO_MINUTES = 15
BACKUP_MINUTES = 4


def worst_case_loss_minutes(interval_minutes):
    return interval_minutes + BACKUP_MINUTES


loss_10 = worst_case_loss_minutes(10)
loss_15 = worst_case_loss_minutes(15)
loss_10_writes = loss_10 * 60 * WRITES_PER_SECOND

print(f"10-minute interval -> {loss_10} minutes, {loss_10_writes:,} writes worst case")
print(f"15-minute interval -> {loss_15} minutes worst case")
print("10-minute interval meets RPO:", loss_10 <= RPO_MINUTES)

assert loss_10 == 14
assert loss_10_writes == 1_680_000
assert loss_10 <= RPO_MINUTES
assert loss_15 > RPO_MINUTES
print("ok")